# JS09 — Lab 2: Exact NN vs. FAISS

**Praktikum Pembelajaran Mesin — Politeknik Negeri Malang**

| | |
|---|---|
| Nama | TBD |
| NIM | TBD |
| Kelas | TBD |
| Tanggal | TBD |

**Tujuan:** membandingkan **IndexFlatL2** (exact) dengan **IndexIVFPQ** (approximate) milik FAISS pada 1.000 titik acak 2D, serta mengukur recall dan waktu pencariannya.

**Prasyarat:** library `faiss` (`faiss-cpu`, dipasang otomatis bila belum ada). Tidak ada dataset eksternal.

> Jalankan di **Google Colab**: `Runtime` ▸ `Run all`. Notebook ini cukup memakai CPU, tidak perlu GPU.


# Lab 2 — Exact NN vs. FAISS

Modul: `JS09 › Lab 2 — Exact NN vs. FAISS`

Percobaan ini membuat pencarian sederhana memakai FAISS. Dua indeks dibandingkan:

| Indeks | Sifat | Keterangan |
|---|---|---|
| `IndexFlatL2` | **exact** | brute force teroptimasi, jarak L2 dihitung penuh |
| `IndexIVFPQ` | **approximate** | klaster IVF (`nlist`) + kompresi Product Quantization (`m` subvektor, 8 bit) |

Data: 1.000 vektor 2D pada rentang [0, 1), query `[0.5, 0.5]`, `k = 3`.


## Langkah 1 — Instal FAISS

Modul menulis dua perintah install: `faiss-cpu` **dan** `faiss-gpu`. Di Colab jangan memasang keduanya sekaligus — paketnya saling menimpa, dan wheel `faiss-gpu` sering tidak tersedia untuk versi Python Colab saat ini. Notebook ini memakai `faiss-cpu`; baris `faiss-gpu` disediakan tetapi dimatikan.

In [ ]:
# Install FAISS
try:
    import faiss  # noqa: F401
except ImportError:
    !pip install -q faiss-cpu
# !pip install -q faiss-gpu   # buka hanya bila Anda benar-benar butuh versi GPU

## Langkah 2 — Import Library

In [ ]:
import numpy as np
import faiss
import matplotlib.pyplot as plt
import time

## Langkah 3 — Membuat Dummy Dataset

FAISS mewajibkan `float32`, jadi data dan query dikonversi sejak awal.

In [ ]:
# Build random data
np.random.seed(42)
X = np.random.rand(1000, 2).astype('float32')  # 1000 data 2D
query = np.array([[0.5, 0.5]], dtype='float32')  # query point

## Langkah 4 — Buat Model Exact NN

`IndexFlatL2` menghitung jarak ke **seluruh** vektor tanpa kompresi maupun clustering, sehingga hasilnya selalu exact nearest neighbor.

In [ ]:
# Search NN using Exact NN
index_flat = faiss.IndexFlatL2(2)   # L2 = Euclidean distance
index_flat.add(X)

# Compute Exact NN Time
start = time.time()
D_flat, I_flat = index_flat.search(query, 3)  # Find 3 NN
end = time.time()
time_flat = end - start

**Catatan:** nilai `D_flat` adalah **jarak kuadrat** L2, bukan jarak Euclidean. Contoh: `0.00013095` setara dengan `sqrt(0.00013095) ≈ 0.01144` pada skala Euclidean — ini perilaku normal FAISS, bukan bug.

## Langkah 5 — Buat Model FAISS

`IVFPQ` menggabungkan dua teknik: **IVF** membagi ruang vektor menjadi `nlist` klaster (karena itu `train()` wajib dipanggil sebelum `add()`), lalu **PQ** memampatkan tiap vektor menjadi `m` subvektor dengan 8 bit per subvektor. Syaratnya `m` harus membagi dimensi (di sini `d = 2`, `m = 2`).

In [ ]:
# Compute FAISS
nlist = 10   # n-cluster (inverted list)
m = 2        # number of subvector
quantizer = faiss.IndexFlatL2(2)   # used by IVF for initiatl cluster
index_ivfpq = faiss.IndexIVFPQ(quantizer, 2, nlist, m, 8)  # 8 bit per subvector

index_ivfpq.train(X)  # training centroid
index_ivfpq.add(X)

start = time.time()
D_ivfpq, I_ivfpq = index_ivfpq.search(query, 3)
end = time.time()
time_ivfpq = end - start

## Langkah 6 — Cek Hasil Performa

In [ ]:
# Print The Peformances
print("Exact NN (Flat) indices:", I_flat, "distances:", D_flat)
print("IVF+PQ indices:", I_ivfpq, "distances:", D_ivfpq)
print("Waktu Exact:", time_flat)
print("Waktu IVF+PQ:", time_ivfpq)

**Contoh output modul:**

```text
Exact NN (Flat) indices: [[137 170 750]] distances: [[0.00013095 0.00077404 0.00079751]]
IVF+PQ indices: [[137 170 750]] distances: [[0.00012945 0.00079226 0.00080067]]
Waktu Exact: 0.00035691261291503906
Waktu IVF+PQ: 0.0001952648162841797
```

Pada dataset kecil ini FAISS hanya sedikit lebih cepat, dan jaraknya berbeda tipis karena PQ memampatkan vektor. Modul juga mengingatkan bahwa data dummy hanya 1.000 baris — keunggulan FAISS baru terasa pada data besar.

In [ ]:
# [tambahan] import untuk tabel rekap
import pandas as pd

In [ ]:
# [tambahan] Jarak FAISS = kuadrat L2 -> akarkan agar sebanding dengan np.linalg.norm
print("Exact (Flat) jarak Euclidean:", np.sqrt(D_flat[0]))
print("IVF+PQ jarak (approx):", np.sqrt(D_ivfpq[0]))

sama = set(int(i) for i in I_flat[0]) & set(int(i) for i in I_ivfpq[0])
print("Irisan indeks:", sorted(sama), f"({len(sama)}/3)")
print("Recall@3:", len(sama) / 3)

### [tambahan] Ukur ulang dengan banyak query

Satu query saja terlalu sensitif terhadap noise waktu. Di bawah ini 200 query acak dipakai, dan recall dihitung terhadap hasil `IndexFlatL2` (exact).

In [ ]:
# [tambahan] 200 query: waktu per query + recall@3
rng = np.random.default_rng(0)
Q = rng.random((200, 2)).astype('float32')

start = time.time()
D_f, I_f = index_flat.search(Q, 3)
t_f = time.time() - start

start = time.time()
D_i, I_i = index_ivfpq.search(Q, 3)
t_i = time.time() - start

def recall_at_k(pred, gt, k=3):
    return float(np.mean([len(set(p[:k]) & set(g[:k])) / k for p, g in zip(pred, gt)]))

print(f"Exact (Flat) : {t_f*1000/len(Q):.4f} ms/query | recall@3 = {recall_at_k(I_f, I_f):.3f}")
print(f"IVF+PQ       : {t_i*1000/len(Q):.4f} ms/query | recall@3 = {recall_at_k(I_i, I_f):.3f}")

### [tambahan] Pengaruh `nprobe`

`nprobe` menentukan berapa klaster IVF yang diperiksa saat query. Semakin besar nilainya, semakin akurat tetapi semakin lambat.

In [ ]:
# [tambahan] Scan nprobe: recall@3 vs waktu
rows = []
for nprobe in [1, 2, 5, 10]:
    index_ivfpq.nprobe = nprobe
    start = time.time()
    _, I = index_ivfpq.search(Q, 3)
    elapsed = (time.time() - start) * 1000 / len(Q)
    rows.append({'nprobe': nprobe,
                 'recall@3': recall_at_k(I, I_f),
                 'ms/query': elapsed})

hasil_nprobe = pd.DataFrame(rows).round(4)
hasil_nprobe

## Langkah 7 — Visualisasi

`I_ivfpq` pada visualisasi ini berasal dari Langkah 5 dengan `nprobe = 1` (nilai default FAISS). Scan `[tambahan]` di atas mengubah `index_ivfpq.nprobe`, tetapi tidak mengubah hasil `I_ivfpq` yang sudah tersimpan.

In [ ]:
# Visualize
plt.scatter(X[:,0], X[:,1], alpha=0.4, label="Dataset")
plt.scatter(query[:,0], query[:,1], c='red', marker='*', s=200, label="Query")

# Flat NN -> Exact NN
plt.scatter(X[I_flat[0],0], X[I_flat[0],1], c='blue', s=100, edgecolor='k', label="Exact NN")

# IVF+PQ NN -> FAISS
plt.scatter(X[I_ivfpq[0],0], X[I_ivfpq[0],1], c='green', marker='x', s=100, label="IVF+PQ NN")

plt.legend()
plt.title("Exact NN vs FAISS IVF+PQ")
plt.show()

**Analisis:** TBD

*(Apakah `I_flat` dan `I_ivfpq` berisi indeks yang sama? Bandingkan `np.sqrt(D_flat)` dengan `np.sqrt(D_ivfpq)`; jelaskan mengapa jarak IVF+PQ sedikit berbeda. Jelaskan juga mengapa `IndexFlatL2` tidak bisa disebut ANN.)*

---
## Kesimpulan Lab 2

**Perbedaan hasil Flat vs IVF+PQ (indeks & jarak):** TBD

**Perbandingan waktu query:** TBD

**Pengaruh `nprobe` terhadap recall dan waktu:** TBD

**Kesimpulan umum:** TBD